## Training a tokenizer from scratch - complete process including regex

Will use gpt-4 regex, 256 merges and taylorswift.txt and support special tokens.

In [6]:
import regex as re
import json

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class JashTokenizer:
    def __init__(self):
        self.regex_pattern = GPT4_SPLIT_PATTERN
        self.compiled_pattern = re.compile(self.regex_pattern)
        self.final_vocab_size = 512
        
        self.vocab = None
        self.merges = None
        self.special_characters = {
            "<|endoftext|>": self.final_vocab_size
        }
   
    def get_stats(self, chunks):
        
        '''
        gets the count of all pair of tokens from all chunks globally
        '''

        counts = {}
        
        for chunk in chunks:
        
            for pair in zip(chunk, chunk[1:]):
                counts[pair] = counts.get(pair, 0) + 1
            
        return counts
    
    def merge(self, ids, pair, idx):
        
        '''
        implements the BPE algorithm, merges the "pair" into a new "idx" in the "ids"
        '''
        
        new_ids = []
        i = 0
        
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
                new_ids.append(idx)
                i += 2
                
            else:
                new_ids.append(ids[i])
                i += 1
                
        return new_ids
                
    def train(self, text):
        
        '''
        "trains" the tokenizer, i.e uses the BPE algorithm to make merges on the tokens and adds to the vocab
        '''
         
        self.vocab = {idx : bytes([idx]) for idx in range(256)}
        self.merges = {}
        
        chunks = self.compiled_pattern.findall(text)
        num_merges = self.final_vocab_size - 256
        
        chunks = [list(ch.encode("utf-8")) for ch in chunks]
        
        for i in range(num_merges):
            
            stats = self.get_stats(chunks)
            max_pair = max(stats, key=stats.get)
            
            idx = (256 + i)
            
            print(f"Merging {max_pair} into new token {idx}")
            
            new_chunk_list = []
            for chunk in chunks:
                new_chunk = self.merge(chunk, max_pair, idx)
                new_chunk_list.append(new_chunk)
                
            self.merges[max_pair] = idx
            
            chunks = new_chunk_list
            
        # now we add to vocab
        
        for (p0, p1), idx in self.merges.items():
            self.vocab[idx] = self.vocab[p0] + self.vocab[p1]
            # byte concatenation
            
        for token, idx in self.special_characters.items():
            self.vocab[idx] = token.encode("utf-8")
                
    def encode(self, text):
        
        '''
        inference function to encode a sentence or text
        '''
        
        if self.merges is None:
            print("Train the Tokenizer to use inference.")
            return
        
        parts = text.split("<|endoftext|>")

        encoded = []

        for i, part in enumerate(parts):

            regex_chunks = self.compiled_pattern.findall(part)

            for ch in regex_chunks:
                chunk = list(ch.encode("utf-8"))

                while True:
                    stats = self.get_stats([chunk])

                    if not stats:
                        break

                    pair = min(
                        stats,
                        key=lambda p: self.merges.get(p, float("inf"))
                    )

                    if pair not in self.merges:
                        break

                    chunk = self.merge(
                        chunk,
                        pair,
                        self.merges[pair]
                    )

                encoded.append(chunk)

            if i < len(parts) - 1:
                encoded.append([self.special_characters["<|endoftext|>"]])

        return encoded
    
    def decode(self, chunks):
        
        '''
        Inference function to decode a list of chunks
        '''
        
        decoded_text_chunks = []
        
        for chunk in chunks:
            
            if len(chunk) == 1 and chunk[0] in self.special_characters.values():
                special_character = next(
                    token for token, idx in self.special_characters.items() if idx == chunk[0]
                )
                
                decoded_text_chunks.append(special_character)
                continue
            
            byte_stream = b"".join(self.vocab[idx] for idx in chunk)
            text = byte_stream.decode("utf-8")
            
            decoded_text_chunks.append(text)
            
        return "".join(decoded_text_chunks)

In [7]:
import os

current_folder = os.getcwd()
data_folder = os.path.join(current_folder, "..", "datasets")
file_path = os.path.join(data_folder, "taylorswift.txt")

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()
    
MinTokenizer = JashTokenizer()
MinTokenizer.train(text)

Merging (101, 114) into new token 256
Merging (50, 48) into new token 257
Merging (111, 114) into new token 258
Merging (105, 110) into new token 259
Merging (101, 100) into new token 260
Merging (32, 116) into new token 261
Merging (111, 110) into new token 262
Merging (104, 101) into new token 263
Merging (32, 83) into new token 264
Merging (97, 114) into new token 265
Merging (97, 110) into new token 266
Merging (32, 65) into new token 267
Merging (261, 263) into new token 268
Merging (97, 108) into new token 269
Merging (114, 105) into new token 270
Merging (118, 260) into new token 271
Merging (115, 116) into new token 272
Merging (119, 105) into new token 273
Merging (32, 82) into new token 274
Merging (257, 49) into new token 275
Merging (32, 102) into new token 276
Merging (257, 50) into new token 277
Merging (32, 84) into new token 278
Merging (102, 116) into new token 279
Merging (97, 121) into new token 280
Merging (32, 34) into new token 281
Merging (273, 279) into new toke

Took ~12 seconds to train the Tokenizer on taylorswift text and 256 new tokens.

In [8]:
sentences_to_test = [
    "Hello, World!",
    "",
    "?",
    "hello world!!!? (안녕하세요!) lol123 😉",
    "",
    "Hello <|endoftext|> world"
]

edge_cases = ["a", "I", " ", "", "3", "!", "😀", "a b", "\n"]

test_text = """The James Webb Space Telescope, launched on December 25, 2021, orbits roughly 1.5 million km from Earth at the Sun-Earth L2 Lagrange point. It cost approximately $10 billion and took over 20 years to develop—delayed multiple times since its original 2007 target date.

Unlike Hubble, JWST observes primarily in the infrared spectrum (0.6–28.5 microns), letting it peer through cosmic dust clouds and detect light from galaxies formed 13.5 billion years ago. Its 6.5-meter primary mirror is made of 18 hexagonal gold-coated beryllium segments.

Scientists weren't expecting SMACS 0723's deep field image to reveal thousands of galaxies in a single frame—some barely 1/10,000,000th as bright as what the human eye can see. NASA's budget for FY2025 allocates $1.2B toward continued operations, don't you think that's a steal for humanity's furthest-reaching eye?
"""

sentences_to_test.append(test_text)

for sentence in sentences_to_test:
    
    print("="*10)
    
    print(sentence)
    
    if MinTokenizer.decode(MinTokenizer.encode(sentence)) == sentence:
        print("Decode -> encode works.")
        
    encoded = MinTokenizer.encode(sentence)
    
    if MinTokenizer.encode(MinTokenizer.decode(encoded)) == encoded:
        print("Encode -> Decode works.")
        
    print("="*10)
    
for case in edge_cases:
    
    print("="*10)
    
    print(case)
    
    if MinTokenizer.decode(MinTokenizer.encode(case)) == case:
        print("Decode -> encode works.")
        
    encoded = MinTokenizer.encode(case)
    
    if MinTokenizer.encode(MinTokenizer.decode(encoded)) == encoded:
        print("Encode -> Decode works.")
        
    print("="*10)
    

    

Hello, World!
Decode -> encode works.
Encode -> Decode works.

Decode -> encode works.
Encode -> Decode works.
?
Decode -> encode works.
Encode -> Decode works.
hello world!!!? (안녕하세요!) lol123 😉
Decode -> encode works.
Encode -> Decode works.

Decode -> encode works.
Encode -> Decode works.
Hello <|endoftext|> world
Decode -> encode works.
Encode -> Decode works.
The James Webb Space Telescope, launched on December 25, 2021, orbits roughly 1.5 million km from Earth at the Sun-Earth L2 Lagrange point. It cost approximately $10 billion and took over 20 years to develop—delayed multiple times since its original 2007 target date.

Unlike Hubble, JWST observes primarily in the infrared spectrum (0.6–28.5 microns), letting it peer through cosmic dust clouds and detect light from galaxies formed 13.5 billion years ago. Its 6.5-meter primary mirror is made of 18 hexagonal gold-coated beryllium segments.

Scientists weren't expecting SMACS 0723's deep field image to reveal thousands of galaxies